# PatchTST Training -- ETTh1

Trains PatchTST (Nie et al., ICLR 2023) at four forecast horizons (96, 192, 336, 720) on the ETTh1 dataset.

**Prerequisites**
- Accelerator set to **GPU T4 x2** (Settings -> Accelerator)
- ETTh1.csv attached as a dataset at `/kaggle/input/etth1-dataset/ETTh1.csv`

**Outputs** (written to `/kaggle/working/results/`)
- `patchtst_pred{N}.csv` -- per-epoch train/val metrics for each horizon
- `checkpoints/patchtst_pred{N}_best.pt` -- best checkpoint per horizon
- `patchtst_ettch1.csv` -- consolidated test MSE/MAE summary

In [ ]:
# Cell 1 -- Environment verification
# Expected: PyTorch 2.x, CUDA available on T4 session.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pandas", "numpy"], check=True)

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("WARNING: no GPU detected. Switch the session accelerator to GPU T4 x2.")

In [ ]:
# Cell 2 -- ETTh1Dataset
from __future__ import annotations

from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

_TRAIN_END: int = 8640
_VAL_END:   int = 11520
_TEST_END:  int = 14400

Split = Literal["train", "val", "test"]


class ETTh1Dataset(Dataset):
    """Sliding window dataset over the ETTh1 time series.

    Split protocol matches PatchTST, iTransformer, and TimeMixer papers:
        Train : rows [0,     8640)
        Val   : rows [8640,  11520)
        Test  : rows [11520, 14400)

    Normalisation: per-channel z-score, scaler fitted on train split only.

    Args:
        csv_path: Path to ETTh1.csv.
        split:    One of 'train', 'val', 'test'.
        seq_len:  Number of input timesteps.
        pred_len: Number of target timesteps immediately following the input.
    """

    def __init__(self, csv_path: str | Path, split: Split, seq_len: int, pred_len: int) -> None:
        super().__init__()
        if split not in ("train", "val", "test"):
            raise ValueError(f"split must be one of 'train', 'val', 'test', got '{split}'")
        if seq_len < 1:
            raise ValueError(f"seq_len must be >= 1, got {seq_len}")
        if pred_len < 1:
            raise ValueError(f"pred_len must be >= 1, got {pred_len}")

        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.split    = split

        raw = self._load_csv(Path(csv_path))
        self._validate_length(raw)

        train_rows = raw[:_TRAIN_END]
        self._mean = train_rows.mean(axis=0)
        self._std  = train_rows.std(axis=0, ddof=0).clip(min=1e-8)
        normalized = (raw - self._mean) / self._std

        start, end    = self._split_bounds(split)
        self._data    = normalized[start:end].astype(np.float32)

        window = seq_len + pred_len
        if len(self._data) < window:
            raise ValueError(
                f"Split '{split}' has {len(self._data)} rows but "
                f"seq_len + pred_len = {window}. Reduce seq_len or pred_len."
            )
        self._num_samples = len(self._data) - window + 1

    def __len__(self) -> int:
        return self._num_samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        if idx < 0 or idx >= self._num_samples:
            raise IndexError(f"Index {idx} out of range [0, {self._num_samples})")
        x = self._data[idx : idx + self.seq_len]
        y = self._data[idx + self.seq_len : idx + self.seq_len + self.pred_len]
        return torch.from_numpy(x), torch.from_numpy(y)

    @property
    def num_features(self) -> int:
        """Number of channels / variates."""
        return self._data.shape[1]

    @staticmethod
    def _load_csv(path: Path) -> np.ndarray:
        if not path.exists():
            raise FileNotFoundError(
                f"ETTh1.csv not found at {path}. "
                "Download from https://github.com/zhouhaoyi/ETDataset"
            )
        df = pd.read_csv(path)
        numeric = df.drop(columns=["date"])
        if numeric.isnull().any().any():
            raise ValueError("ETTh1.csv contains NaN values; preprocessing required.")
        return numeric.to_numpy(dtype=np.float64)

    @staticmethod
    def _validate_length(data: np.ndarray) -> None:
        if len(data) < _TEST_END:
            raise ValueError(
                f"ETTh1.csv has {len(data)} rows but at least {_TEST_END} are required "
                "for the standard 12/4/4 month split."
            )

    @staticmethod
    def _split_bounds(split: Split) -> tuple[int, int]:
        bounds: dict[str, tuple[int, int]] = {
            "train": (0,          _TRAIN_END),
            "val":   (_TRAIN_END, _VAL_END),
            "test":  (_VAL_END,   _TEST_END),
        }
        return bounds[split]


# Smoke test
_ds = ETTh1Dataset.__new__(ETTh1Dataset)
print("ETTh1Dataset defined.")

In [ ]:
# Cell 3 -- PatchTST model
# Reference: Nie et al., "A Time Series Is Worth 64 Words", ICLR 2023.
# https://arxiv.org/abs/2211.14730

import math
import torch
import torch.nn as nn


class PatchEmbedding(nn.Module):
    """Extract overlapping patches from a univariate series and project to d_model.

    Args:
        patch_size: Number of time steps per patch.
        stride:     Stride between consecutive patches.
        d_model:    Projection dimension.
        dropout:    Dropout rate applied after embedding.
    """

    def __init__(self, patch_size: int, stride: int, d_model: int, dropout: float) -> None:
        super().__init__()
        self.patch_size = patch_size
        self.stride     = stride
        self.projection = nn.Linear(patch_size, d_model)
        self.dropout    = nn.Dropout(dropout)

    @staticmethod
    def _sinusoidal_encoding(num_patches: int, d_model: int, device: torch.device) -> torch.Tensor:
        position = torch.arange(num_patches, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float, device=device)
            * (-math.log(10000.0) / d_model)
        )
        encoding = torch.zeros(1, num_patches, d_model, device=device)
        encoding[0, :, 0::2] = torch.sin(position * div_term)
        encoding[0, :, 1::2] = torch.cos(position * div_term[: d_model // 2])
        return encoding

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, seq_len, 1)
        Returns:
            (B, num_patches, d_model)
        """
        x       = x.squeeze(-1)
        pad_len = self.stride - ((x.size(1) - self.patch_size) % self.stride)
        if pad_len < self.stride:
            x = torch.nn.functional.pad(x, (0, pad_len))
        x = x.unfold(dimension=1, size=self.patch_size, step=self.stride)
        x = self.projection(x)
        x = x + self._sinusoidal_encoding(x.size(1), x.size(2), x.device)
        return self.dropout(x)


class _TransformerBlock(nn.Module):
    """Pre-norm transformer block: LayerNorm -> attention -> residual,
    LayerNorm -> MLP -> residual."""

    def __init__(self, d_model: int, num_heads: int, mlp_ratio: int = 4, dropout: float = 0.0) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp   = nn.Sequential(
            nn.Linear(d_model, d_model * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * mlp_ratio, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed      = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)
        x           = x + attn_out
        x           = x + self.mlp(self.norm2(x))
        return x


class TransformerEncoder(nn.Module):
    """Stack of pre-norm transformer blocks with a final layer norm.

    Args:
        d_model:    Model dimension.
        num_heads:  Number of attention heads.
        num_layers: Number of stacked transformer blocks.
        dropout:    Dropout rate.
    """

    def __init__(self, d_model: int, num_heads: int, num_layers: int, dropout: float) -> None:
        super().__init__()
        self.blocks = nn.ModuleList(
            [_TransformerBlock(d_model, num_heads, dropout=dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for block in self.blocks:
            x = block(x)
        return self.norm(x)


class ForecastHead(nn.Module):
    """Flatten patch tokens and project to a forecast horizon.

    Args:
        num_patches: Number of patch tokens.
        d_model:     Model dimension.
        pred_len:    Forecast horizon.
    """

    def __init__(self, num_patches: int, d_model: int, pred_len: int) -> None:
        super().__init__()
        self.flatten = nn.Flatten(start_dim=1)
        self.linear  = nn.Linear(num_patches * d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(self.flatten(x))


class PatchTST(nn.Module):
    """Channel-independent PatchTST for multivariate long-horizon forecasting.

    Channels are processed independently: the input is reshaped from (B, seq_len, C)
    to (B*C, seq_len, 1) before the encoder. Weights are shared across channels by
    construction. This is the PatchTST/64 configuration from Nie et al., ICLR 2023.

    Args:
        seq_len:      Input sequence length.
        pred_len:     Forecast horizon.
        num_variates: Number of input channels.
        patch_size:   Time steps per patch. Default 16 (paper config).
        stride:       Stride between patches. Default 8 (paper config).
        d_model:      Transformer model dimension. Default 128 (paper config).
        num_heads:    Number of attention heads. Default 16 (paper config).
        num_layers:   Number of transformer blocks. Default 3 (paper config).
        dropout:      Dropout rate. Default 0.2 (paper config).
    """

    def __init__(
        self,
        seq_len:      int,
        pred_len:     int,
        num_variates: int,
        patch_size:   int   = 16,
        stride:       int   = 8,
        d_model:      int   = 128,
        num_heads:    int   = 16,
        num_layers:   int   = 3,
        dropout:      float = 0.2,
    ) -> None:
        super().__init__()
        self.num_variates = num_variates
        self.pred_len     = pred_len

        pad_len     = stride - ((seq_len - patch_size) % stride)
        padded_len  = seq_len + (pad_len if pad_len < stride else 0)
        num_patches = (padded_len - patch_size) // stride + 1

        self.patch_embedding = PatchEmbedding(patch_size, stride, d_model, dropout)
        self.encoder         = TransformerEncoder(d_model, num_heads, num_layers, dropout)
        self.head            = ForecastHead(num_patches, d_model, pred_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, seq_len, C)
        Returns:
            (B, pred_len, C)
        """
        B, seq_len, C = x.shape
        x = x.permute(0, 2, 1).reshape(B * C, seq_len, 1)
        x = self.patch_embedding(x)
        x = self.encoder(x)
        x = self.head(x)
        x = x.reshape(B, C, self.pred_len).permute(0, 2, 1)
        return x


# Shape smoke test
_x   = torch.randn(2, 512, 7)
_out = PatchTST(seq_len=512, pred_len=96, num_variates=7)(_x)
assert _out.shape == (2, 96, 7), f"Unexpected shape: {_out.shape}"
del _x, _out
print("PatchTST defined and shape verified.")

In [ ]:
# Cell 4 -- Training infrastructure
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

KAGGLE_CONFIG: dict = {
    "data_path":    Path("/kaggle/input/etth1-dataset/ETTh1.csv"),
    "output_dir":   Path("/kaggle/working/results"),
    "seq_len":      512,
    "batch_size":   128,
    "lr":           1e-4,
    "weight_decay": 1e-4,
    "grad_clip":    1.0,
    "epochs":       100,
    "warmup_ratio": 0.1,
    "patience":     10,
    "patchtst": {
        "patch_size": 16,
        "stride":     8,
        "d_model":    128,
        "num_heads":  16,
        "num_layers": 3,
        "dropout":    0.2,
    },
    "seed": 42,
}

# ---------------------------------------------------------------------------
# Early stopping
# ---------------------------------------------------------------------------


class EarlyStopping:
    """Stop training when validation MSE has not improved for `patience` epochs.

    Args:
        patience: Number of epochs to wait after the last improvement.
    """

    def __init__(self, patience: int) -> None:
        self.patience  = patience
        self._best:    float = float("inf")
        self._counter: int   = 0

    @property
    def best(self) -> float:
        return self._best

    def step(self, val_mse: float) -> bool:
        """Return True if training should stop."""
        if val_mse < self._best:
            self._best    = val_mse
            self._counter = 0
            return False
        self._counter += 1
        return self._counter >= self.patience


# ---------------------------------------------------------------------------
# Metric computation
# ---------------------------------------------------------------------------


def compute_metrics(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
) -> tuple[float, float]:
    """Compute MSE and MAE over a data loader.

    Uses reduction='sum' and divides by total elements to give the true
    dataset-level mean regardless of variable last-batch size.

    Args:
        model:  Model in eval mode.
        loader: DataLoader yielding (x, y) pairs.
        device: Compute device.

    Returns:
        Tuple of (mse, mae).
    """
    total_mse = 0.0
    total_mae = 0.0
    total_n   = 0

    with torch.no_grad():
        for x, y in loader:
            x, y  = x.to(device), y.to(device)
            pred  = model(x)
            n     = y.numel()
            total_mse += nn.functional.mse_loss(pred, y, reduction="sum").item()
            total_mae += nn.functional.l1_loss(pred, y, reduction="sum").item()
            total_n   += n

    return total_mse / total_n, total_mae / total_n


print("Training infrastructure defined.")

In [ ]:
# Cell 5 -- train_patchtst function
def train_patchtst(pred_len: int) -> dict:
    """Train PatchTST for one forecast horizon on ETTh1.

    Saves per-epoch metrics to results/patchtst_pred{pred_len}.csv and the
    best checkpoint (by validation MSE) to results/checkpoints/.
    Loads the best checkpoint and evaluates on the test split before returning.

    Args:
        pred_len: Forecast horizon in time steps.

    Returns:
        Dict with keys 'pred_len', 'test_mse', 'test_mae'.
    """
    seed = KAGGLE_CONFIG["seed"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seq_len   = KAGGLE_CONFIG["seq_len"]
    data_path = Path(KAGGLE_CONFIG["data_path"])

    print(f"\n{'=' * 60}")
    print(f"PatchTST | pred_len={pred_len} | device={device}")
    print(f"{'=' * 60}")

    train_ds = ETTh1Dataset(data_path, split="train", seq_len=seq_len, pred_len=pred_len)
    val_ds   = ETTh1Dataset(data_path, split="val",   seq_len=seq_len, pred_len=pred_len)
    test_ds  = ETTh1Dataset(data_path, split="test",  seq_len=seq_len, pred_len=pred_len)

    num_variates = train_ds.num_features

    train_loader = DataLoader(train_ds, batch_size=KAGGLE_CONFIG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=KAGGLE_CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=KAGGLE_CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

    cfg   = KAGGLE_CONFIG["patchtst"]
    model = PatchTST(
        seq_len=seq_len,
        pred_len=pred_len,
        num_variates=num_variates,
        patch_size=cfg["patch_size"],
        stride=cfg["stride"],
        d_model=cfg["d_model"],
        num_heads=cfg["num_heads"],
        num_layers=cfg["num_layers"],
        dropout=cfg["dropout"],
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {total_params:,}")

    optimiser = torch.optim.AdamW(
        model.parameters(),
        lr=KAGGLE_CONFIG["lr"],
        weight_decay=KAGGLE_CONFIG["weight_decay"],
    )

    epochs        = KAGGLE_CONFIG["epochs"]
    warmup_epochs = max(1, int(epochs * KAGGLE_CONFIG["warmup_ratio"]))
    scheduler     = torch.optim.lr_scheduler.SequentialLR(
        optimiser,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(
                optimiser, start_factor=1e-3, end_factor=1.0, total_iters=warmup_epochs
            ),
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimiser, T_max=epochs - warmup_epochs, eta_min=1e-6
            ),
        ],
        milestones=[warmup_epochs],
    )

    early_stopping  = EarlyStopping(patience=KAGGLE_CONFIG["patience"])
    criterion       = nn.MSELoss()

    output_dir      = Path(KAGGLE_CONFIG["output_dir"])
    checkpoint_dir  = output_dir / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    run_id          = f"patchtst_pred{pred_len}"
    csv_path        = output_dir / f"{run_id}.csv"
    checkpoint_path = checkpoint_dir / f"{run_id}_best.pt"

    with open(csv_path, "w", newline="") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["epoch", "train_mse", "val_mse", "train_mae", "val_mae", "lr"])

        for epoch in range(1, epochs + 1):
            # Train
            model.train()
            train_mse_sum = 0.0
            train_mae_sum = 0.0
            train_n       = 0

            for x, y in train_loader:
                x, y  = x.to(device), y.to(device)
                optimiser.zero_grad()
                pred  = model(x)
                loss  = criterion(pred, y)

                if torch.isnan(loss):
                    raise RuntimeError(f"NaN loss at epoch {epoch}. Check data normalisation.")

                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), KAGGLE_CONFIG["grad_clip"])
                optimiser.step()

                n              = y.numel()
                train_mse_sum += loss.item() * n
                train_mae_sum += nn.functional.l1_loss(pred.detach(), y, reduction="sum").item()
                train_n       += n

            scheduler.step()

            train_mse  = train_mse_sum / train_n
            train_mae  = train_mae_sum / train_n

            # Validate
            model.eval()
            val_mse, val_mae = compute_metrics(model, val_loader, device)
            current_lr       = optimiser.param_groups[0]["lr"]

            writer.writerow([
                epoch,
                f"{train_mse:.6f}",
                f"{val_mse:.6f}",
                f"{train_mae:.6f}",
                f"{val_mae:.6f}",
                f"{current_lr:.2e}",
            ])
            csv_file.flush()

            print(
                f"Epoch {epoch:3d}/{epochs} | "
                f"train_mse={train_mse:.4f}  val_mse={val_mse:.4f} | "
                f"train_mae={train_mae:.4f}  val_mae={val_mae:.4f} | "
                f"lr={current_lr:.2e}"
            )

            if val_mse < early_stopping.best:
                torch.save(model.state_dict(), checkpoint_path)

            if early_stopping.step(val_mse):
                print(f"Early stopping at epoch {epoch} (best val_mse={early_stopping.best:.4f}).")
                break

    # Evaluate on test split using the best checkpoint
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    test_mse, test_mae = compute_metrics(model, test_loader, device)

    targets = {96: (0.370, 0.400), 192: (0.413, 0.422), 336: (0.422, 0.440), 720: (0.447, 0.468)}
    t_mse, t_mae = targets[pred_len]
    print(f"\nTest | pred_len={pred_len} | MSE={test_mse:.4f}  MAE={test_mae:.4f}")
    print(f"Paper | pred_len={pred_len} | MSE={t_mse:.3f}   MAE={t_mae:.3f}")

    return {"pred_len": pred_len, "test_mse": test_mse, "test_mae": test_mae}


print("train_patchtst defined.")

In [ ]:
# Cell 6 -- Run all four horizons sequentially
# Expected wall time on T4: ~25-35 min per horizon, ~2 hours total.
results = []
for pred_len in [96, 192, 336, 720]:
    metrics = train_patchtst(pred_len)
    results.append(metrics)

In [ ]:
# Cell 7 -- Consolidate per-horizon test results into patchtst_ettch1.csv
import csv
from pathlib import Path

output_dir   = Path(KAGGLE_CONFIG["output_dir"])
summary_path = output_dir / "patchtst_ettch1.csv"

with open(summary_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["pred_len", "test_mse", "test_mae"])
    writer.writeheader()
    writer.writerows(results)

print(f"Summary saved to {summary_path}\n")

published = {96: (0.370, 0.400), 192: (0.413, 0.422), 336: (0.422, 0.440), 720: (0.447, 0.468)}
print(f"{'pred_len':>10} {'MSE (ours)':>12} {'MAE (ours)':>12} {'MSE (paper)':>13} {'MAE (paper)':>13} {'MSE gap':>10}")
print("-" * 75)
for r in results:
    pl             = int(r["pred_len"])
    p_mse, p_mae   = published[pl]
    gap            = r["test_mse"] - p_mse
    print(
        f"{pl:>10} {r['test_mse']:>12.4f} {r['test_mae']:>12.4f} "
        f"{p_mse:>13.3f} {p_mae:>13.3f} {gap:>+10.4f}"
    )

In [ ]:
# Cell 8 -- Verify all output files are present before closing the session
from pathlib import Path

output_dir     = Path(KAGGLE_CONFIG["output_dir"])
checkpoint_dir = output_dir / "checkpoints"

expected = (
    [output_dir / f"patchtst_pred{p}.csv"            for p in [96, 192, 336, 720]]
    + [checkpoint_dir / f"patchtst_pred{p}_best.pt"  for p in [96, 192, 336, 720]]
    + [output_dir / "patchtst_ettch1.csv"]
)

all_present = True
for path in expected:
    status = "OK" if path.exists() else "MISSING"
    if status == "MISSING":
        all_present = False
    print(f"  [{status}] {path}")

if not all_present:
    raise RuntimeError("Some output files are missing. Do not close the session.")

print("\nAll outputs verified. Safe to download and close session.")